# Prediction Routine

In [1]:
import pandas as pd
from tensorflow.keras.models import load_model  # type: ignore
from tensorflow.keras.models import model_from_json  # type: ignore

from deep.modelling.custom_loss import CategoricalFocalLoss
from deep.predict.model_fetcher import fetch_model
from deep.preprocess.image_cleaner import clean_test_directory 
from deep.preprocess.resize_inputs import resize_test_album 
from deep.predict.predict_main import predict_images_from_dir
from deep.constants import MODEL_DICT, MODEL_IMAGE_SIZE, CLEANER_JSONS, MODEL_CONFIGS


2025-05-01 13:28:58.254559: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-01 13:28:58.255195: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-01 13:28:58.258206: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-01 13:28:58.265330: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746102538.278301    8200 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746102538.28

In [10]:
# Prediction pipeline args
image_dir = "Path/to/your/images" # provide a valid directory with images
cleaner_config_path = CLEANER_JSONS / "" # point to a valid cleaning json
model = input(f"Select a model from {list(MODEL_DICT.keys())}: ") # select one of our models

In [ ]:
# Cleaner
clean_test_directory(config_path=cleaner_config_path, input_dir=image_dir)

In [ ]:
# Resizer
resize_test_album(model=MODEL_DICT[model], path=image_dir)

In [ ]:
# Get Model weights
weights_path = fetch_model(model=model)

# Get Model Json
json_path = MODEL_CONFIGS / "model_expanded_architecture_2025-04-30.json"
custom_objects = {"CategoricalFocalLoss": CategoricalFocalLoss}

from tensorflow.keras.models import model_from_json
import os

#Load the model architecture from the JSON file
with open(os.path.join(MODEL_CONFIGS, 'model_base_architecture.json'), 'r') as json_file:
    model_json = json_file.read()

model = model_from_json(model_json, custom_objects={'CategoricalFocalLoss': loss})

#Load the model weights
model.load_weights(os.path.join(MODELS, 'model_base_weights.weights.h5'))

Model already exists at ./EXPANDED_MODEL_01.h5, skipping download.


TypeError: <class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': {'optimizer': {'module': 'keras.optimizers', 'class_name': 'RMSprop', 'config': {'name': 'rmsprop', 'learning_rate': 0.0001250000059371814, 'weight_decay': None, 'clipnorm': None, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': False, 'ema_momentum': 0.99, 'ema_overwrite_frequency': None, 'loss_scale_factor': None, 'gradient_accumulation_steps': None, 'rho': 0.9, 'momentum': 0.0, 'epsilon': 1e-07, 'centered': False}, 'registered_name': None}, 'loss': {'module': 'deep.modelling.custom_loss', 'class_name': 'CategoricalFocalLoss', '

In [ ]:
# Make predictions
df_preds = predict_images_from_dir(model_instance, image_dir, MODEL_IMAGE_SIZE[MODEL_DICT[model]])
df_preds.head()

In [ ]:
# Store the results
df_preds.to_csv()